# P1 session A - validate before spending anything

Everything here is cheap. It exists so that a session B or C is never started on a corpus or a task that cannot support the paper.\n\nRun this first, read the last cell, and only continue if it says the gate passed.

**Settings: Accelerator `GPU T4 x2`, Internet `ON`.**

Nothing in this notebook configures the experiment. Every cell runs a script
from the repo; the design lives in `configs/experiment.yaml` and
`configs/p1_split_manifest.json`. If a check fails, stop and report it -- the
design is not adjusted to make a check pass.

In [ ]:
# 1. Get the code.
REPO_URL = "https://github.com/fairuz-anadi/quantization.git"
REF      = "main"          # pin to a commit SHA for the run that goes in the paper

import os, subprocess, sys
SRC = "/kaggle/working/quantlang"
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO_URL, SRC],
                   check=True)
print(subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())
os.chdir(SRC); sys.path.insert(0, SRC)

In [ ]:
# 2. Dependencies. Kaggle's torch is CUDA-matched -- never reinstall it.
!pip install -q -U "transformers>=4.45" "bitsandbytes>=0.43" "peft>=0.13" accelerate datasets pyyaml

# torchao is REMOVED, not upgraded. Kaggle ships torchao 0.10.0; a
# current PEFT wants >= 0.16.0, and its is_torchao_available() RAISES
# on an out-of-range version instead of returning False. PEFT probes it
# for every LoRA layer it builds, so with both installed no adapter can
# attach at all and fine-tuning cannot run.
#
# This pipeline never uses torchao -- INT8 and NF4 are both bitsandbytes.
# Upgrading it instead could pull a different torch, which is the one
# thing on Kaggle that must not move.
!pip uninstall -q -y torchao

In [ ]:
# 3. Environment probe. Raises and STOPS the notebook if this session cannot
#    run the experiment: a P100 (no INT8/NF4), or a PEFT/torchao mismatch
#    that makes LoRA attachment impossible.
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/probe_env.py", "--outdir", "/kaggle/working")

## The frozen contracts

`freeze_p0.py` proves P0 is byte-identical to what produced the published
results. `pytest` covers the P1 construction, including the checks that version
1 did not have: the substring shortcut is worth exactly a guess, the gold and
the distractors are present at the same rate, and the FT arm is verified to
differ from the Base arm.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/freeze_p0.py")
gate("-m", "pytest", "-q")

## The corpus reproduces exactly

Re-derives every P1 item from the pinned dataset revision, the frozen
`split_seed` and the pinned tokenizer, and compares against the frozen digests.
Downloads the corpus, so it takes a few minutes.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/build_p1_splits.py", "--check")

## The learnability gate

Two questions, cheapest first.

1. **CPU.** Does the frozen item set still carry a lexical shortcut? Version 1
   scored ~0.96 (English) and ~0.92 (Bangla) on "choose the option that appears
   verbatim in the passage", on a 100% gold-presence rate. Version 2 scores
   exactly 0.25 because all four options are present.
2. **GPU, a few minutes.** How well does the BASE model already do on the P1
   task, scored by the exact P0 evaluator? The threshold is P0's own best
   measured cell, read from `results/ALL_P0_RESULTS/tables/accuracy.csv` -- it
   is not a number chosen here.

   Measured 2026-08-29: **0.970 English, 0.900 Bangla**, so English trips it.
   This is a WARNING, passed with `--acknowledge-low-headroom` and recorded in
   the report. It bounds what the FT arm can show on ACCURACY -- RQ3 is expected
   to be null for English -- but it does not make the FT arm vacuous: check 9
   below measures the FT model against the base at matched precision and found a
   1.14 logit delta from three optimizer steps, against 0.000000 for the invalid
   v1 run. Empty headroom and an unchanged model are different claims.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/check_p1_learnability.py", *"--langs eng_Latn ben_Beng".split(), "--outdir", "/kaggle/working/p1_gate", "--acknowledge-low-headroom")

## The 20-item smoke test

Nine checks. The ninth is new: it compares the fine-tuned logits against the
base model's. A full English run once passed checks 1-8 while producing logits
bit-identical to the base model at every precision, because nothing compared the
two arms.

Read the **fp16** row of `max_logit_delta_vs_base_fp16`: the baseline is the base
model at FP16, so only that row isolates fine-tuning. The int8 and nf4 rows carry
the quantization effect too and rise for that reason alone.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/run_p1_smoke.py", "--outdir", "/kaggle/working/p1_smoke", "--lang", "eng_Latn")

In [ ]:
# Read the report.
import json
r = json.load(open("/kaggle/working/p1_smoke/p1_smoke_report.json", encoding="utf-8"))
for name, check in sorted(r["checks"].items()):
    print(f"[{'PASS' if check['pass'] else 'FAIL'}] {name}")

d = r["checks"]["9_ft_arm_differs_from_base_arm"]
print("\nFT vs base, max logit delta (fp16 row is the comparable one):")
print("   ", d["max_logit_delta_vs_base_fp16"])
print("merge weight delta:", d["merge_weight_delta"])
print("\nALL CHECKS PASSED:", r["all_checks_passed"])

## Gate

Continue to session B only if:

* `freeze_p0.py` reports the P0 freeze intact (the 30 unregistered raw
  provenance files are a known pre-existing gap, not a failure);
* `pytest` is green;
* `--check` reports the split reproduces exactly;
* the learnability gate prints `GATE PASSED`;
* all nine smoke checks read PASS, and check 9's logit delta is **not** 0.0.

Then download `/kaggle/working/p1_gate/p1_learnability_report.json` and
`/kaggle/working/p1_smoke/p1_smoke_report.json`.

In [ ]:
import shutil, os
KEEP = "/kaggle/working/p1_sessionA_keep"
os.makedirs(KEEP, exist_ok=True)
for src in ["/kaggle/working/p1_gate/p1_learnability_report.json",
            "/kaggle/working/p1_smoke/p1_smoke_report.json"]:
    if os.path.exists(src):
        shutil.copy(src, KEEP)
if os.path.isdir("/kaggle/working/p1_smoke/adapter"):
    shutil.copytree("/kaggle/working/p1_smoke/adapter", f"{KEEP}/adapter",
                    dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/p1_sessionA", "zip", KEEP)

# The merged checkpoint is ~5.75 GB and is DERIVED -- rebuildable from the base
# model plus the adapter. Freeing it keeps the session inside the 20 GB limit.
if os.path.isdir("/kaggle/working/p1_smoke/merged"):
    shutil.rmtree("/kaggle/working/p1_smoke/merged")
print(sorted(os.listdir("/kaggle/working")))